# Análise das Transcrições das Lives do Frei Gilson - Quaresma 2025

## Introdução

**Autor:** *Bruno Conterato*

**Objetivo:** *Analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025 utilizando técnicas modernas de processamento de linguagem natural (NLP).*

---

## Etapas de Pré-processamento
Antes da análise, o notebook prepara as transcrições para que o conteúdo fique mais organizado e fácil de processar.

### O que é feito
1. **Limpeza básica do texto**
   - remove espaços extras no início e no fim de cada linha;
   - padroniza a formatação da transcrição.

2. **Divisão em blocos menores**
   - separa a transcrição em chunks de tamanho controlado;
   - mantém uma sobreposição entre blocos para preservar contexto.

3. **Identificação de referências bíblicas**
   - detecta quando há citação explícita de um livro, capítulo e versículo;
   - extrai essas referências em formato estruturado.

4. **Filtro de conteúdo relevante**
   - orienta o modelo a considerar apenas o que foi dito durante o Rosário;
   - ignora a reflexão final e outros trechos que não fazem parte da análise principal.

---

## Necessidades de Processamento
Para analisar corretamente as transcrições, o notebook precisa lidar com alguns tipos de conteúdo diferentes:

1. **Separar os momentos da fala**
   - identificar o ponto de divisão entre as reflexões do Terço e as reflexões do Dia;
   - separar os textos em duas partes: reflexão do Terço e reflexão do Dia.

2. **Filtrar trechos que não entram na análise principal**
   - remover as seções marcadas com a tag `[Música]`;
   - identificar e remover orações oficiais da Igreja Católica.

3. **Registrar músicas citadas durante a live**
   - extrair e documentar as músicas cantadas;
   - registrar o nome da música e o autor de cada uma.

4. **Contextualizar os ensinamentos dentro do Rosário**
   - considerar em que momento do Rosário cada ensinamento foi apresentado;
   - relacionar cada trecho ao terço e ao mistério meditado naquele instante.

---

## Recursos Externos Utilizados
Além do texto do notebook, este processo depende de alguns recursos que ficam fora dele, mesmo quando estão no mesmo projeto local:

1. **Vector Store da Bíblia**
   - armazenado em `../Bíblia VectorStore/biblia_vectorstore`;
   - usado para buscar passagens bíblicas semelhantes aos trechos analisados.

2. **Banco SQLite da Bíblia**
   - acessado em `../Bíblia VectorStore/biblia.db`;
   - contém os versículos consultados durante a extração das referências.

3. **Modelo de embeddings**
   - `sentence-transformers/all-MiniLM-L6-v2`;
   - transforma trechos de texto em vetores para permitir a busca por similaridade.

4. **Modelo de linguagem local**
   - executado via `Ollama`;
   - gera as respostas e faz a extração estruturada das informações.

5. **Arquivos de entrada e saída do projeto**
   - leitura em `../../data/raw/Santo Rosário | Quaresma 2025/Youtube to Text`;
   - saída prevista em `../../data/processed/Santo Rosário | Quaresma 2025/Youtube to Text`.


## 1. Configurações

### 1.1. Importação de Bibliotecas


In [1]:
import os
from typing import List, Optional


import pandas as pd
import sqlite3
from tqdm.notebook import tqdm
from pprint import pprint

from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.output_parsers.enum import EnumOutputParser, Enum
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers.pydantic import PydanticOutputParser
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field, field_validator, ValidationError
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain.output_parsers import RetryOutputParser
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model
from langchain.output_parsers import OutputFixingParser
from langchain_core.exceptions import OutputParserException

from langchain.tools import tool

from langgraph.graph import MessagesState
from langchain.chat_models import init_chat_model

from utils import BookEnum, ORDERED_BOOKS, BinaryResponse

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
MODEL_PROVIDER = "ollama"
# MODEL_PROVIDER = "google_genai"

# MODEL = "Gemma-3-Gaia-PT-BR-4b-it-f16:latest"
MODEL = "batiai/gemma4-e4b:q4"

# List of Models: https://ai.google.dev/gemini-api/docs/models
# MODEL = "gemini-2.0-flash"

### 1.2. Hiperparâmetros

**Chunk size / Overlap**


Tabela: percentis da quantidade de caracteres por versículo bíblico

| Métrica | Valor Associado |
| :--- | :--- |
| **Percentil p0** | 2 |
| **Percentil p10** | 32 |
| **Percentil p25 (Q1)** | 65 |
| **Percentil p50 (Mediana)** | 94 |
| **Percentil p75 (Q3)** | 135 |
| **Percentil p90** | 176 |
| **Percentil p95** | 202 |
| **Percentil p99** | 256 |
| **Percentil p100** | 576 |


In [3]:
MIN_SIMILARITY_THRESHOLD = 0.65
CHUNK_SIZE = 5000
CHUNK_OVERLAP = 600

## 2.0. Processamento de texto

In [4]:
def trim_line_whitespace(text):
    """
    Remove todos os tabs e espaços em branco do início e do final de cada linha do texto.
    """
    lines = text.split("\n")
    cleaned_lines = [line.strip() for line in lines]
    return "\n".join(cleaned_lines)

In [5]:
system_message = """
    Você é um especialista na fé católica.  
    Receberá um **texto com a transcrição da oração do Santo Rosário**, rezado ao vivo pelo Frei Gilson durante um dia da Quaresma de 2025.

    A oração está dividida em duas partes:
    - A primeira parte é o Santo Rosário, onde Frei Gilson reza o Rosário com os fiéis.
    - A segunda parte é uma reflexão relacionada à fé católica e ao silêncio.

    ⚠️ **IMPORTANTE:**  
    Ignore qualquer parte da transcrição que seja da parte da **reflexão**.  
    Use **apenas o que for dito durante a oração do Rosário**.

    ---

    ### Sua missão é extrair e organizar as informações nos itens de 1 a 5 abaixo:

    ---

    ## 1. Temática principal

    - Identifique a principal ideia que Frei Gilson ensina enquanto reza o Rosário.
    - Resuma em até 3 parágrafos.
    - Use somente ensinamentos que ele falou **durante a oração do Santo Rosário**.

    ---

    ## 2. Temáticas secundárias

    - Identifique de 2 a 5 temas que Frei Gilson também falou **durante a oração do Santo ROsário**.
    - Para cada tema:
    - Dê um título claro (ex: *Perseverança na fé*).
    - Escreva 1 a 2 parágrafos com os ensinamentos relacionados.

    ---

    ## 3. Versículos da Bíblia

    Vou lhe fornecer os **versículos bíblicos citados durante a oração**.
    Você só ignorará versículos que sejam referentes a orações como o Pai nosso, Ave Maria, Credo etc,
    mas manterá todos os demais versículos.
    Não se esqueça de nenhum versículo ou intervalo de versículos que eu lhe fornecer, a menos que citado antyeriormente nas excessões.
    Traga apenas versículos que foram citados durante a oração do Santo Rosário.
    Traga a localização do versículo, a transcrição integral do versículo e os ensinamentos que foram transmitidos por aquele versículo e pela narração do Frei Gilson.
    Para cada um:

    - Escreva assim:  
    `(Livro Capítulo, Versículo)`: <Transcrição integral do versículo> ou
    `(Livro Capítulo, Versículo-Início–Versículo-Fim)`: <Transcrição integral do intervalo de versículos>

    - Logo abaixo, escreva:
    **Ensinamentos:**: <Parágrafo ou frase que explique o que o Frei falou sobre esse versículo durante a oração>

    Se o versículo for relacionado a algum mistério do Santo Rosário (como `Segundo Mistério Doloroso` ou `Quinto Mistério Gozozo`), diga isso.

    ---

    ## 4. Músicas

    Se Frei Gilson mencionar músicas durante a oração:

    - Diga o nome da música e o artista, como neste exemplo:
    <Nome da música> - <Artista>: <contexto>  

    - Depois escreva o que Frei Gilson disse sobre a música.

    ---

    ## 5. Eventos de Agenda

    Se Frei Gilson mencionar **eventos** (missas, encontros, lives etc.):
    Traga todos os eventos citados durante a oração ou ao final do Santo Rosário.
    - Traga o nome do evento, a data e o local.
    - Depois escreva o que ele falou sobre esse evento.

    ---

    ## ⚠️ Regras finais:

    - NÃO copie orações conhecidas (Pai Nosso, Ave Maria, Credo etc.).
    - Não explique o rosário, exceto se dor necessário para compreender o respectivo ensinamento.
    - NÃO traga informações da reflexão final que ocorre após o rosário. Traga só o que é dito durante o Rosário.
    - NÃO invente nada. Só use o que estiver escrito.
    - NÃO deixe seções em branco. Se algo não aparecer, **remova a seção inteira**.
    ```
"""

## 3.0 Detecção de trechos da bíblia

In [6]:
# 0. Carregue a transcrição (demo)
# transcription = """
# Hoje refletimos sobre a importância de sermos humildes. Como está escrito: "Bem-aventurados os humildes, pois herdarão a terra".
# Mais adiante, mencionou-se que devemos amar o próximo como a nós mesmos.
# """

# 1. Divida a transcrição
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
)
# chunks = splitter.create_documents([transcription])

# 2. Carregue os embeddings e a base vetorial
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
db = Chroma(
    collection_name="biblia",
    persist_directory="../Bíblia VectorStore/biblia_vectorstore",
    embedding_function=embedding_model,
)


# 3. Defina a enum e o parser
binary_parser = EnumOutputParser(enum=BinaryResponse)


# 4. Modelo Pydantic para saída estruturada de referência bíblica
class BibleReference(BaseModel):
    book: BookEnum = Field(
        ...,
        description="Nome do livro da Bíblia explicitamente anunciado. Não deve conter número do capítulo.",
        examples=ORDERED_BOOKS,
    )
    chapter: int = Field(
        ...,
        description="Número do capítulo explicitamente anunciado. Não deve conter número de versículos.",
    )
    verse_start: int = Field(
        ...,
        description="Número do primeiro versículo do intervalo explicitamente anunciado, se for um único versículo. Se for um intervalo, é o primeiro versículo do intervalo.",
    )
    verse_end: int = Field(
        ...,
        description="Número do último versículo do intervalo explicitamente anunciado, se for um intervalo. Se não houver intervalo, igual ao verse_start.",
    )

    @field_validator("verse_end", mode="before")
    @classmethod
    def set_verse_end(cls, v, values):
        if v is None:
            return values.get("verse_start")
        return v


class BibleReferences(BaseModel):
    bible_excerpts: List[BibleReference] = Field(
        ..., description="Lista com todos os trechos da bíblia identificados"
    )


# Parser para saída estruturada
bible_reference_parser = JsonOutputParser(pydantic_object=BibleReferences)

# Novo prompt: pede saída JSON estruturada para referência bíblica
# bible_reference_prompt_template = """
# Você é um assistente que analisa trechos de transcrição de fala e identifica qual é a **passagem bíblica** citada de forma clara e explícita.

# Sua tarefa é identificar e **extrair** as seguintes informações, mesmo que existam variações de linguagem falada ou erros de transcrição:
# - O **nome do livro da Bíblia**
# - O **número do capítulo**
# - O **número do(s) versículo(s)** — pode ser um único versículo (ex: 3) ou um intervalo (ex: 3-5)

# Sua resposta deve ser exatamente neste formato **sem explicações, comentários ou qualquer outro texto**:
# {format_instructions}

# Texto a ser analisado:
# {query}
# """

# bible_reference_prompt = PromptTemplate(
#     template=bible_reference_prompt_template,
#     input_variables=["query"],
#     partial_variables={
#         "format_instructions": bible_reference_parser.get_format_instructions()
#     },
# )

# print("Format Instructions:")
# print(bible_reference_parser.get_format_instructions())
# 5. LLM local
# Funciona bem com os modelos instruct:
# - llama3.1:8b-instruct-q5_K_M
# - mistral:7b-instruct
llm = ChatOllama(model=MODEL)

# 6. Cadeia completa manual (RAG customizado)


# def format_context(docs: List[Document], show_option=False):
#     if show_option:
#         return "\n\n".join(
#             f"Excerpt {i + 1}: {doc.metadata['book']} {doc.metadata['chapter']}:{doc.metadata['verse']} {doc.page_content}"
#             for (i, doc) in enumerate(docs)
#         )
#     return "\n\n".join(
#         f"{doc.metadata['book']} {doc.metadata['chapter']}:{doc.metadata['verse']} {doc.page_content}"
#         for doc in docs
#     )


@tool
def retriever_bible_passages_tool(
    query: str, k: int = 3, min_similarity_threshould: float = MIN_SIMILARITY_THRESHOLD
):
    """
    Recupera as passagens bíblicas mais similares ao trecho fornecido.
    """
    retriever = db.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": k, "score_threshold": min_similarity_threshould},
    )
    retriever_top_k = retriever.invoke(query)[:k]
    print("Retriever top k: ", retriever_top_k)
    return retriever_top_k


# def retrieve_books_and_chapters(docs: List[Document]) -> str:
#     """
#     Retrieve all verses from the specific book and chapter found in docs.
#     Assumes a table 'versiculos' with columns: id, book, chapter, verse, text, line_number.
#     """
#     conn = sqlite3.connect("../Bíblia VectorStore/biblia.db")
#     cursor = conn.cursor()
#     results = []
#     seen = set()
#     for doc in docs:
#         book = doc.metadata.get("book")
#         chapter = doc.metadata.get("chapter")
#         if book and chapter and (book, chapter) not in seen:
#             seen.add((book, chapter))
#             results.append(f"\n\nLivro: {book} - Capítulo: {chapter}")
#             cursor.execute(
#                 "SELECT book, chapter, verse, text FROM versiculos WHERE book=? AND chapter=? ORDER BY verse ASC",
#                 (book, chapter),
#             )
#             for row in cursor.fetchall():
#                 results.append(f"{row[2]}: {row[3]}")
#     conn.close()
#     return "\n".join(results) if results else "Nenhum resultado encontrado."

In [7]:
# Pipeline binário para detecção de referência bíblica
# binary_bible_reference_template = """
# Você é um assistente que analisa trechos de transcrição de fala e identifica se há uma **citação bíblica clara e explícita**.

# Sua tarefa é:
# 1. Verificar se o trecho contém uma **anunciação clara** de que será citada uma **passagem bíblica**.
# 2. A citação deve conter, de forma explícita (mesmo com erros de transcrição ou variações da fala):
#    - O **nome do livro da Bíblia**
#    - O **número do capítulo**
#    - O **número do(s) versículo(s)** — pode ser um único versículo (ex: 3) ou um intervalo (ex: 3-5)

# Se você encontrar essa citação bíblica clara, responda apenas com "Sim.". Caso contrário, responda apenas com "Não.".
# Não escreva nenhuma explicação, justificativa ou comentário.
# O resultado deve ser apenas "Sim." ou "Não." — nada além disso.

# Texto a ser analisado:
# {query}
# """

# binary_bible_reference_prompt = PromptTemplate(
#     template=binary_bible_reference_template,
#     input_variables=["query"],
# )

# binary_bible_reference_chain = (
#     {"query": lambda x: x["query"]}
#     | binary_bible_reference_prompt
#     | llm
#     | binary_parser
# )
# binary_bible_reference_tool = binary_bible_reference_chain.as_tool()

In [8]:
fixing_parser = OutputFixingParser.from_llm(parser=bible_reference_parser, llm=llm)

# bible_reference_chain = {"query": lambda x: x["query"]} | bible_reference_prompt | llm

llm_with_tools = llm.bind_tools([retriever_bible_passages_tool])


BIBLE_EXCERPTS_JOIN_PROMPT = """
Retorne os trechos da bíblia abaixo, juntando se forem contínuos no mesmo capítulo do mesmo livro:

Trechos:
{bible_excerpts}
"""


def find_bible_versicles(state: MessagesState, verbose=False):
    """Call the model to generate a response based on the current state. Given
    the question, it will decide to retrieve using the retriever tool, or simply respond to the user.
    """
    print("[generate_query_or_respond] invoking: ", state["messages"])
    ai_msg = llm_with_tools.invoke(state["messages"])
    if not ai_msg.tool_calls:
        return {"messages": [ai_msg]}

    tool_messages = []
    for tc in ai_msg.tool_calls:
        if verbose:
            print("\nTool call: ", tc)
        tool_result = retriever_bible_passages_tool.invoke(tc["args"]["query"])
        if verbose:
            print("\nTool call response: ", tool_result)
        tool_messages.append(
            {
                "role": "tool",
                "tool_call_id": tc["id"],
                "content": str(tool_result),
            }
        )

    final_msg = llm.invoke(
        BIBLE_EXCERPTS_JOIN_PROMPT.format(bible_excerpts=tool_messages)
    )
    if verbose:
        print("\nFinal message content:\n", final_msg.content)
    parsed = fixing_parser.parse(final_msg.content)
    return {"messages": [parsed]}


EXTRACT_BIBLE_PASSAGES_SYSTEM_PROMPT = """
Você é um teólogo renomado e conhecido por conseguir identificar trechos da bíblia.

Você deve identificar e isolar trechos nos textos a seguir que possam conter versículos da bíblia, um candidato a versículo por trecho.
Envie cada trecho isolado individualmente para a ferramenta `retriever_bible_passages_tool` e colete a resposta.

Uma dica para inferir início dos versículos: uso de expressões como: "E disse o Senhor a Moisés..." ou "Então disse Jesus..."
ou uma citação explícita do livro, capítulo e versículo, como "Livro de Gênesis, capítulo 1, versículo 1"
ou "Livro de João, capítulo 3, versículos 16 a 17".

Uma dica para inferir fim dos versículos: quando houver uma mudança de assunto, ou quebra da sequência lógica do texto,
ou quando houver uma mudança de locutor, ou quando houver uma mudança de contexto.

As dicas podem ou não valer, cada caso deve ser analisado individualmente, mas são boas pistas para identificar trechos contíguos
que possam conter versículos da bíblia.

Atenção: O trecho pode conter mais de um versículo, parágrafo ou frase.
Envie à ferramenta cada trecho contíguo que houver possibilidade de conter um trecho contíguo da bíblia.
"""


def get_bible_passages_from_chunk_text(text: str, verbose=False):

    input = {
        "messages": [
            {"role": "system", "content": EXTRACT_BIBLE_PASSAGES_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": text,
            },
        ]
    }
    bible_passages = find_bible_versicles(input, verbose)["messages"][-1]
    if verbose:
        print("\nPassagens da bíblia: ", bible_passages)
    return bible_passages


def get_all_bible_passages(transcription: str, verbose: bool = False) -> list:
    """
    Extrai referências bíblicas de uma transcrição, com prints opcionais para depuração.
    """
    bible_passages = []
    chunks = splitter.create_documents([transcription])
    if verbose:
        print(f"[get_bible_passages] Total de chunks: {len(chunks)}")

    for idx, chunk in enumerate(tqdm(chunks, desc="Processando chunks")):
        if verbose:
            print(f"\n[get_bible_passages] Chunk {idx + 1}/{len(chunks)}:")
            print(f"\nConteúdo do chunk:\n\n{chunk.page_content}\n")

        response = get_bible_passages_from_chunk_text(
            chunk.page_content, verbose=verbose
        )
        if verbose:
            print("\nLLM response:")
            print(response)

        try:
            refs = BibleReferences.model_validate(response)
            bible_passages.append(refs)
        except ValidationError as e:
            print(f"Erro ao validar referência: {e}")
    
    # Flatten all BibleReferences into a list of individual BibleReference objects  
    all_refs = []
    for br in bible_passages:
        if hasattr(br, 'bible_excerpts'):
            all_refs.extend(br.bible_excerpts)
        elif isinstance(br, dict) and 'bible_excerpts' in br:
            all_refs.extend(br['bible_excerpts'])
        else:
            all_refs.append(br)
    
    # Sort by book order, chapter, verse_start
    def sort_bible_references(refs: list) -> list:
        """Sort a list of BibleReference objects or dicts by book order, chapter, verse_start."""
        book_order = {book.value: idx for idx, book in enumerate(ORDERED_BOOKS)}
        def sort_key(ref):
            if hasattr(ref, 'book'):
                book = ref.book
                chapter = ref.chapter if ref.chapter is not None else 0
                verse_start = ref.verse_start if ref.verse_start is not None else 0
            else:
                book = ref.get('book', '')
                chapter = ref.get('chapter', 0)
                verse_start = ref.get('verse_start', 0)
            return (book_order.get(book, 999), chapter, verse_start)
        return sorted(refs, key=sort_key)

    all_refs = sort_bible_references(all_refs)
    bible_passages = all_refs
    
    if verbose:
        print(
            f"\n[get_bible_passages] Total de referências extraídas: {len(bible_passages)}"
        )
    # Alternativa simples: usar pandas para remover duplicatas de dicionários

    # Se os itens forem dicionários ou BaseModel, converta para dicionário
    dicts = [
        passage.dict() if hasattr(passage, "dict") else passage
        for passage in bible_passages
    ]
    unique_passages = pd.DataFrame(dicts).drop_duplicates().to_dict(orient="records")
    return unique_passages


# Exemplo de uso:
# Propositalmente, chamamos o livro errado.
# Livro correto: 1 Corintios
bible_refs = get_all_bible_passages(
    transcription=trim_line_whitespace(
        """
        Livro de Primeiro João, capítulo 3, versículos 16 a 17.
        
        Não sabeis que sois o Templo de Deus, e que o Espírito de Deus habita em vós?
        Se alguém destruir o Templo de Deus, Deus o destruirá. 
        Porque o templo de Deus é sagrado – e isso sois vós.
        """
    ),
    verbose=True,
)
print("Referências bíblicas extraídas:")
for ref in bible_refs:
    print(ref)

[get_bible_passages] Total de chunks: 1


Processando chunks:   0%|          | 0/1 [00:00<?, ?it/s]


[get_bible_passages] Chunk 1/1:

Conteúdo do chunk:

Livro de Primeiro João, capítulo 3, versículos 16 a 17.

Não sabeis que sois o Templo de Deus, e que o Espírito de Deus habita em vós?
Se alguém destruir o Templo de Deus, Deus o destruirá.
Porque o templo de Deus é sagrado – e isso sois vós.

[generate_query_or_respond] invoking:  [{'role': 'system', 'content': '\nVocê é um teólogo renomado e conhecido por conseguir identificar trechos da bíblia.\n\nVocê deve identificar e isolar trechos nos textos a seguir que possam conter versículos da bíblia, um candidato a versículo por trecho.\nEnvie cada trecho isolado individualmente para a ferramenta `retriever_bible_passages_tool` e colete a resposta.\n\nUma dica para inferir início dos versículos: uso de expressões como: "E disse o Senhor a Moisés..." ou "Então disse Jesus..."\nou uma citação explícita do livro, capítulo e versículo, como "Livro de Gênesis, capítulo 1, versículo 1"\nou "Livro de João, capítulo 3, versículos 16 a 17".\n\n

/tmp/ipykernel_1785901/2857616937.py:151: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  passage.dict() if hasattr(passage, "dict") else passage


## 4.0. Leitura dos Arquivos

In [9]:
model = init_chat_model(MODEL, model_provider=MODEL_PROVIDER)

raw_folder = "../../data/raw/Santo Rosário | Quaresma 2025/Youtube to Text"
processed_folder = "../../data/processed/Santo Rosário | Quaresma 2025/Youtube to Text"
titulo_template = (
    "Santo Rosário | Quaresma 2025 | 03:40 | {ordem}° Dia | Live Ao vivo.txt"
)

print("Diretório atual:", os.getcwd())


def gerar_titulo_fonte(ordem):
    return titulo_template.format(ordem=ordem)


for i in tqdm(range(13, 14), desc="Processando arquivos"):
    titulo_source = gerar_titulo_fonte(str(i))
    arquivo = f"{raw_folder}/{titulo_source}"
    titulo_md = f"{titulo_source[:-3]}.md"

    if os.path.exists(arquivo):
        with open(arquivo, "r+", encoding="utf-8") as f:
            conteudo = f.read()
            if conteudo:
                # TODO: retirar limites
                conteudo = conteudo[:20000]  # Limitar a 20.000 caracteres inicialmente
                print(f"\n\nArquivo: {titulo_source}")

                bible_refs = get_all_bible_passages(conteudo, verbose=True)
                print("Referências bíblicas extraídas:")
                for ref in bible_refs:
                    print(ref)

                messages = [
                    {"role": "system", "content": trim_line_whitespace(system_message)},
                    {"role": "user", "content": conteudo},
                ]

                chunks = []
                for text in model.stream(messages):
                    chunks.append(text.text())
                    print(text.text(), end="", flush=True)
                response = "".join(chunks)

                # with open(
                #     f"{processed_folder}/rosario_{titulo_md}", "w+", encoding="utf-8"
                # ) as f:
                #     f.write(response)

            else:
                print(f"\n\nO arquivo {titulo_source} está vazio.")
    else:
        print(f"\n\nArquivo não encontrado: {arquivo}")

    break

Diretório atual: /home/bruno/Workspace/MariaGPT/src/Rosários Quaresma Frei Gilson 2025


Processando arquivos:   0%|          | 0/1 [00:00<?, ?it/s]



Arquivo: Santo Rosário | Quaresma 2025 | 03:40 | 13° Dia | Live Ao vivo.txt
[get_bible_passages] Total de chunks: 6


Processando chunks:   0%|          | 0/6 [00:00<?, ?it/s]


[get_bible_passages] Chunk 1/6:

Conteúdo do chunk:

(144) Santo Rosário | Quaresma 2025 | 03:40 | 13° Dia | Live Ao vivo - YouTube
https://www.youtube.com/watch?v=5zDuvPECDoA

[generate_query_or_respond] invoking:  [{'role': 'system', 'content': '\nVocê é um teólogo renomado e conhecido por conseguir identificar trechos da bíblia.\n\nVocê deve identificar e isolar trechos nos textos a seguir que possam conter versículos da bíblia, um candidato a versículo por trecho.\nEnvie cada trecho isolado individualmente para a ferramenta `retriever_bible_passages_tool` e colete a resposta.\n\nUma dica para inferir início dos versículos: uso de expressões como: "E disse o Senhor a Moisés..." ou "Então disse Jesus..."\nou uma citação explícita do livro, capítulo e versículo, como "Livro de Gênesis, capítulo 1, versículo 1"\nou "Livro de João, capítulo 3, versículos 16 a 17".\n\nUma dica para inferir fim dos versículos: quando houver uma mudança de assunto, ou quebra da sequência lógica do texto,\

...


Tool call:  {'name': 'retriever_bible_passages_tool', 'args': {'query': 'Creio em Deus Pai todo poderoso criador do céu e da terra e em Jesus Cristo seu único filho nosso Senhor que foi concebido pelo poder do Espírito Santo nasceu da Virgem Maria padeceu sob Pilatos foi crucificado morto e sepultado desceu àção dos Mortos ressuscitou ao terceiro dia subiu aos céus está sentada à direita de Deus Pai todo poderoso onde há de vir a julgar os vivos e os mortos creio no espírito santo na Santa Igreja Católica na comunhão dos Santos Na remissão dos pecados na ressurreição da carne e na vida eterna Amém'}, 'id': '9f6bfa6f-90b7-43c8-8992-c32680cb1923', 'type': 'tool_call'}
Retriever top k:  []

Tool call response:  []

Final message content:
 Não foi fornecido nenhum trecho bíblico para serem juntados. Por favor, inclua os textos que você gostaria que eu revisasse e combinasse.

Passagens da bíblia:  {'bible_excerpts': []}

LLM response:
{'bible_excerpts': []}

[get_bible_passages] Chunk 5/6

No relevant docs were retrieved using the relevance score threshold 0.65



Tool call:  {'name': 'retriever_bible_passages_tool', 'args': {'query': 'José era um homem de bem não querendo difamar resolveu rejeitá-la secretamente é claro que José conhecia bem as leis da sua época ele sabia que uma mulher que fosse pega em adultério Ela poderia ser levada até a morte ele escolheu o amor ele escolheu a desrição aqui não é a fuga aqui é o amor aqui é a desrição aqui é um homem de bem não quer difamar Olha que bonito isso José não quer difamar ele não entende a situação ele tá confuso ele não sabe o que está acontecendo ele escolhe não'}, 'id': '4a8a29cd-af77-4b23-9951-cd2731fefd3b', 'type': 'tool_call'}
Retriever top k:  []

Tool call response:  []

Final message content:
 Por favor, forneça os trechos da Bíblia que você gostaria que eu combinasse.

O conteúdo enviado está vazio (`[]`), então não consigo identificar quais passagens devo juntar. Assim que você fornecer os versículos ou referências, farei o agrupamento para você!

Passagens da bíblia:  {'bible_excer

No relevant docs were retrieved using the relevance score threshold 0.65
No relevant docs were retrieved using the relevance score threshold 0.65
No relevant docs were retrieved using the relevance score threshold 0.65



Tool call:  {'name': 'retriever_bible_passages_tool', 'args': {'query': 'mistério segundo mistério gozoso nós contemplamos a visitação de Nossa Senhora a sua prima Santa Isabel e nessa hora o espírito santo encheu a vida de Isabel diz a palavra que Isabel ficou cheia do espírito santo'}, 'id': 'ab3792b2-171e-4c9f-a090-432327b21165', 'type': 'tool_call'}
Retriever top k:  []

Tool call response:  []

Tool call:  {'name': 'retriever_bible_passages_tool', 'args': {'query': 'porque a palavra de Deus diz que o espírito é o consolador O Espírito Santo é aquele que consola vem consolar agora espírito santo tantos corações Vem agora espírito santo consolar tantas vidas Vem Espírito Santo ao meu redor eu procuro entender vem espírito santo vem espírito santo e vem me abraçar e vem me abraçar'}, 'id': '73938c82-a2bc-4bb0-91c9-168d2576ea72', 'type': 'tool_call'}
Retriever top k:  []

Tool call response:  []

Tool call:  {'name': 'retriever_bible_passages_tool', 'args': {'query': 'Jesus é aquele 